# Part B · NB5 — Final Comparison (Label-Efficiency vs Part A)  ·  `pavecrack1300-partb-comparison`

**Course:** CSE 438 — Digital Image Processing · EWU · Summer 2026 · **Group:** _`<fill>`_

Pulls each SSL method's `results_partB.json` **and** the Part-A `results.json`, then builds the **label-efficiency**
table + chart (Task E) and consolidated error analysis (Task F). Attach all four method outputs **and** the Part-A
NB1/NB2/NB3 output (for its `results.json`).

In [ ]:
import glob, json
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
WORK = Path("/kaggle/working")
# merge all Part-B method results
pb = {}
for rp in glob.glob("/kaggle/input/**/results_partB.json", recursive=True):
    try: pb.update(json.load(open(rp)))
    except Exception as e: print("skip", rp, e)
# Part-A supervised baseline (best model = DeepLabV3)
partA = {}
for rp in glob.glob("/kaggle/input/**/results.json", recursive=True):
    try: partA.update(json.load(open(rp)))
    except Exception: pass
baseline = max(partA.values(), key=lambda r: r.get("miou", 0)) if partA else None
print("Part-B methods:", list(pb.keys()))
print("Part-A baseline:", baseline["model"] if baseline else "NOT FOUND — attach Part-A output")

## Task E — label-efficiency comparison table

In [ ]:
rows = []
N_LAB = next(iter(pb.values()))["n_labelled"] if pb else 195
for name in sorted(pb, key=lambda k: -pb[k]["miou"]):
    r = pb[name]
    rows.append(dict(Method=name, Backbone="ResNet-50" if name!="DINOv2" else "ViT-S/14",
                     Labels=r["n_labelled"], mIoU=round(r["miou"],4), CrackIoU=round(r["crack_iou"],4),
                     Dice=round(r["mdice"],4), Params_M=r["params_M"]))
if baseline:
    rows.append(dict(Method="Part-A supervised (DeepLabV3)", Backbone="ResNet-50",
                     Labels=909, mIoU=round(baseline["miou"],4), CrackIoU=round(baseline.get("crack_iou",baseline["per_class_iou"][1]),4),
                     Dice=round(baseline.get("mdice",float('nan')),4), Params_M=baseline["params_M"]))
tbl = pd.DataFrame(rows).set_index("Method")
if baseline:
    base_miou = baseline["miou"]
    tbl["%_of_supervised"] = (100*tbl["mIoU"]/base_miou).round(1)
tbl.to_csv(WORK/"partB_label_efficiency.csv"); print(f"Fine-tuned on {N_LAB} labels vs 909 for Part-A ({100*N_LAB/909:.0f}%)")
tbl

## Task E — chart: SSL mIoU vs the supervised baseline

In [ ]:
order = [m for m in ["SimCLR","BYOL","MAE","DINOv2"] if m in pb]
mious = [pb[m]["miou"] for m in order]
fig, ax = plt.subplots(figsize=(8,4.5))
b = ax.bar(order, mious, color=["#4C72B0","#DD8452","#55A868","#C44E52"][:len(order)])
ax.bar_label(b, fmt="%.3f")
if baseline:
    ax.axhline(baseline["miou"], ls="--", color="k", label=f"Part-A supervised ({baseline['miou']:.3f}, 909 labels)")
ax.set_ylabel("test mIoU"); ax.set_ylim(0,1); ax.legend()
ax.set_title(f"Label-efficiency: SSL fine-tuned on {N_LAB} labels vs supervised on 909")
plt.tight_layout(); plt.savefig(WORK/"partB_label_efficiency.png", dpi=120, bbox_inches="tight"); plt.show()

## Task F — consolidated error analysis & label-efficiency discussion
Best SSL method (highest test mIoU) recovers the largest share of the fully-supervised performance while using
only ~21% of the labels. Per-image crack-IoU (stored per method) lets us check whether the SSL models fail on the
same images as the Part-A supervised models — see the report's Insights section.

In [ ]:
if pb:
    best = max(pb, key=lambda k: pb[k]["miou"])
    print(f"Best SSL method: {best} (mIoU {pb[best]['miou']:.4f})")
    if baseline:
        print(f"= {100*pb[best]['miou']/baseline['miou']:.1f}% of the Part-A supervised mIoU, using {N_LAB}/909 = {100*N_LAB/909:.0f}% of the labels")
    print("\nRanking by label-efficiency (test mIoU):")
    for m in sorted(pb, key=lambda k:-pb[k]['miou']): print(f"  {m:10s} mIoU {pb[m]['miou']:.4f} | crackIoU {pb[m]['crack_iou']:.4f}")

**NB5 complete.** Saved the label-efficiency table + chart. These feed report §6.6 (Results) and §6.8 (Label-Efficiency).